# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` Python library.

### Dataset Source
The dataset is defined with a Croissant schema, accessible at:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

This dataset covers structured clinical, laboratory, and genetic features for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset package and explore metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Authors (@id): {[author['@id'] for author in metadata.author]}")
print(f"Keywords: {metadata.keywords}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and column `@id`s.

In Croissant, all entities are referenced by their `@id`. Below we list all available record sets and their structures using their unique identifiers.

In [ ]:
# Retrieve all record sets defined in the dataset
record_sets = dataset.record_sets()

if len(record_sets) == 0:
    print("No record sets defined directly in the dataset metadata.\nAttempting to discover record sets from the schema package.")

    # In mlcroissant, the metadata can be .to_json()'ed for inspection
    package = dataset.metadata.to_json()
    # Try to parse recordSets from the metadata
    # sometimes they are under 'recordSet', sometimes under 'hasPart'
    if 'recordSet' in package:
        record_sets_raw = package['recordSet']
    elif 'hasPart' in package:
        record_sets_raw = package['hasPart']
    else:
        record_sets_raw = []
    
    # Collect @id from each record set
    record_set_ids = [r['@id'] if isinstance(r, dict) else r for r in record_sets_raw]
else:
    record_set_ids = [r['@id'] for r in record_sets]

print("Available Record Sets by @id:")
for ridx, rsid in enumerate(record_set_ids):
    print(f"[{ridx}] {rsid}")

# For each record set, inspect available fields/columns
for rsid in record_set_ids:
    print(f"\nFields in record set {rsid}:")
    # mlcroissant allows inspection of record set fields
    try:
        fields = dataset.fields(record_set=rsid)
        for f in fields:
            print(f"Field: {f['name']} | @id: {f['@id']} | DataType: {f.get('dataType', 'unknown')}")
    except Exception as e:
        print(f"Could not inspect fields for {rsid}: {e}")


## 3. Data Extraction
Load all available record sets into DataFrames for analysis. Use record set and field `@id`s identified above.

You may use the `dataframes` dict for easy reference to each loaded table by its `@id`.

In [ ]:
dataframes = {}

for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded DataFrame for record set {rsid} with shape {df.shape}")
        print("Columns (fields) by @id:")
        print(df.columns.tolist())
        print(df.head(2), "\n")
    except Exception as e:
        print(f"Could not load data for {rsid}: {e}")

# Choose representative record set for downstream analysis
selected_record_set = record_set_ids[0] if len(record_set_ids) > 0 else None
print(f"\nSelected Record Set for EDA: {selected_record_set}")


## 4. Exploratory Data Analysis (EDA)
Apply common processing steps—filtering, normalization, grouping—on fields referenced by their `@id`.

We will use a numeric clinical field and group by a categorical feature (by `@id`). Be sure to update the `numeric_field_id` and `group_field_id` to ones listed above.

In [ ]:
# Example choice: using columns from Table 1 (clinical features) if available

# List fields for selected record set
df = dataframes[selected_record_set]
print(f"Columns in {selected_record_set}: {df.columns.tolist()}")

# Choose a numeric field, e.g., 'age_at_diagnosis', which may be referenced by its @id
# Similarly, choose a categorical group field, e.g., 'infertility_status', referenced by @id

# As an example, we'll attempt to extract relevant @ids
numeric_field_id = None
group_field_id = None

fields = dataset.fields(record_set=selected_record_set)
for f in fields:
    # Looking for a field with int/float dataType
    dt = f.get('dataType', '').lower()
    if dt in ['integer', 'float', 'number']:
        numeric_field_id = f['@id']
        print(f"Numeric field candidate: {f['name']} | @id: {f['@id']} | DataType: {dt}")
    if dt in ['boolean', 'text', 'string']:
        group_field_id = f['@id']
        print(f"Grouping field candidate: {f['name']} | @id: {f['@id']} | DataType: {dt}")
    if numeric_field_id and group_field_id:
        break

# Fallback to column name if @id is not present
if numeric_field_id is None:
    numeric_field_id = df.columns[0]
if group_field_id is None:
    group_field_id = df.columns[1] if len(df.columns) > 1 else df.columns[0]

# Filtering numeric field and normalizing
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by the categorical field
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not present in DataFrame columns.")


## 5. Visualization
Visualize distributions and relationships between fields, referenced by their `@id`.

Below is an example using matplotlib to visualize the distribution of the numeric clinical field and a scatter plot against the group field (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric clinical feature
if numeric_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=5, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id} (by @id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Scatter plot if group_field is available
if group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=group_field_id, y=numeric_field_id, data=df, hue=group_field_id)
    plt.title(f"{numeric_field_id} vs {group_field_id} (by @id)")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()


## 6. Conclusion
This notebook demonstrated loading, overview, extraction, and initial analysis of the FAIR^2 dataset via `mlcroissant`, referencing all entities by their `@id`. Key steps included:
- Loading metadata and inspecting authors, license, and keywords
- Listing available record sets and their fields using Croissant `@id`
- Extracting tabular data and analyzing clinical numeric features
- Visualizing data distributions

With full `@id` referencing, you can confidently trace and manipulate specific fields, record sets, and columns for reproducible research.

**Note:** The FAIR^2 dataset is a case-based, small cohort resource useful for research illustration and analysis of genotype-phenotype correlations in female non-classical 21-hydroxylase deficiency infertility.
